# Trích xuất feature ONE-PEACE cho UniAV (YouCookII)

**Chỉ cần sửa ô số 2 (ĐƯỜNG DẪN).** Các ô sau dùng lại biến ở đó.

Thứ tự: 1 → 2 → 3, sau đó chạy phần **A (visual)** hoặc **B (audio)**. Mỗi lần Colab khởi động lại phải chạy lại 1 → 2 → 3 và bước cài đặt của phần tương ứng.

Runtime → Change runtime type → **GPU (khuyên dùng A100)**.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. ĐƯỜNG DẪN — SỬA Ở ĐÂY cho khớp với chỗ bạn upload trên Drive
ONE_PEACE_DIR = '/content/drive/MyDrive/KL/One_Peace'          # thư mục chứa các file .py này + 2 checkpoint
VIDEO_DIR     = '/content/drive/MyDrive/KL/YouCookII/videos'   # thư mục chứa <video_id>.mp4
FEAT_DIR      = '/content/drive/MyDrive/KL/feats/youcookii'    # nơi ghi *_one_peace_video_finetune.npy và *_one_peace_audio.npy
ANNO_JSON     = f'{ONE_PEACE_DIR}/annotations/youcookii_all.json'  # danh sách video cần xử lý

# Tham số (giữ nguyên để sát bài báo)
STRIDE      = 8              # bước visual theo frame @16fps: 8 = 0.5 s (như ActivityNet), 4 = 0.25 s
STRIDE_SEC  = STRIDE / 16    # bước audio tương ứng, KHÔNG sửa riêng
BATCH_VIDEO = 4              # số clip / lần forward (giảm nếu hết VRAM)
BATCH_AUDIO = 64             # số cửa sổ 1 s / lần forward

# Chia việc cho nhiều phiên Colab: phiên thứ k đặt SHARD_ID = k (0 .. NUM_SHARDS-1)
NUM_SHARDS = 1
SHARD_ID   = 0

In [ ]:
# 3. Kiểm tra đường dẫn + GPU
import os, glob
for p in [ONE_PEACE_DIR, VIDEO_DIR, ANNO_JSON,
          f'{ONE_PEACE_DIR}/onepeace_video_k400.pth', f'{ONE_PEACE_DIR}/one-peace-audio.pt']:
    print('OK   ' if os.path.exists(p) else 'THIẾU', p)
print('số video mp4:', len(glob.glob(f'{VIDEO_DIR}/*.mp4')))
os.makedirs(FEAT_DIR, exist_ok=True)
%cd {ONE_PEACE_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## A. Visual (Python mặc định của Colab)

In [ ]:
# A1. Cài đặt + copy checkpoint ra ổ local của Colab (nạp nhanh hơn đọc từ Drive)
!pip install -q einops
!cp -n "{ONE_PEACE_DIR}/onepeace_video_k400.pth" /content/

In [ ]:
# A2. Kiểm tra nhanh: clip giây 92 phải ra 'making a sandwich'; in thêm sai khác fp16 vs fp32
!python sanity_check_video.py --checkpoint /content/onepeace_video_k400.pth \
    --video "{VIDEO_DIR}/GLd3aX16zBg.mp4" --start_sec 92 --compare_fp16

In [ ]:
# A3. Đo tốc độ trên 200 clip (ghi ra /content/bench, không đụng FEAT_DIR)
!python extract_video_features.py --checkpoint /content/onepeace_video_k400.pth \
    --video_dir "{VIDEO_DIR}" --output_dir /content/bench --ids_from "{ANNO_JSON}" \
    --stride {STRIDE} --batch_size {BATCH_VIDEO} --limit 1 --max_clips 200 --overwrite

In [ ]:
# A4. Chạy thật. Bị ngắt thì chạy lại ô này: video đã có file .npy sẽ được bỏ qua
!python extract_video_features.py --checkpoint /content/onepeace_video_k400.pth \
    --video_dir "{VIDEO_DIR}" --output_dir "{FEAT_DIR}" --ids_from "{ANNO_JSON}" \
    --stride {STRIDE} --batch_size {BATCH_VIDEO} --num_shards {NUM_SHARDS} --shard_id {SHARD_ID}

## B. Audio (cần môi trường Python 3.10 riêng cho fairseq cũ của ONE-PEACE)

In [ ]:
# B1. Cài đặt (~3-5 phút)
!git clone -q --depth 1 https://github.com/OFA-Sys/ONE-PEACE /content/ONE-PEACE || true
!pip install -q uv && uv venv -q --seed --python 3.10 /content/op310
!/content/op310/bin/python -m pip install -q "pip==24.0"
!/content/op310/bin/python -m pip install -q torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
!/content/op310/bin/python -m pip install -q "numpy<2" hydra-core==1.0.7 omegaconf==2.0.6 antlr4-python3-runtime==4.8 \
    bitarray sacrebleu tabulate regex timm==0.6.11 iopath tensorboardX pydub librosa==0.10.0 soundfile soxr einops \
    opencv-python-headless scipy tqdm pillow imageio-ffmpeg
!cp -n "{ONE_PEACE_DIR}/one-peace-audio.pt" /content/

In [ ]:
# B2. Chạy thật. Bị ngắt thì chạy lại ô này
!/content/op310/bin/python extract_audio_features.py --onepeace_repo /content/ONE-PEACE \
    --checkpoint /content/one-peace-audio.pt \
    --video_dir "{VIDEO_DIR}" --output_dir "{FEAT_DIR}" --ids_from "{ANNO_JSON}" \
    --stride_sec {STRIDE_SEC} --batch_size {BATCH_AUDIO} --num_shards {NUM_SHARDS} --shard_id {SHARD_ID}

## C. Kiểm tra kết quả (sau khi xong cả A và B)

In [ ]:
!python check_features.py --anno "{ANNO_JSON}" --feat_dir "{FEAT_DIR}" --stride {STRIDE}